### List all the item in the blocklist

In [1]:
import os
from azure.ai.contentsafety import BlocklistClient
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("CONTENT_SAFETY_KEY")
endpoint = os.getenv("CONTENT_SAFETY_ENDPOINT")

# Create a Blocklist client
client = BlocklistClient(endpoint, AzureKeyCredential(key))

blocklist_name = "blocklist1"

try:
    blocklist_items = client.list_text_blocklist_items(blocklist_name=blocklist_name)
    if blocklist_items:
        print("\nList blocklist items: ")
        for blocklist_item in blocklist_items:
            print(
                f"BlocklistItemId: {blocklist_item.blocklist_item_id}, Text: {blocklist_item.text}, "
                f"Description: {blocklist_item.description}"
            )
except HttpResponseError as e:
    print("\nList blocklist items failed: ")
    if e.error:
        print(f"Error code: {e.error.code}")
        print(f"Error message: {e.error.message}")
        raise
    print(e)
    raise


List blocklist items: 
BlocklistItemId: 8865328d-9b9e-4474-8c67-c282ba65be4c, Text: ^(?:4[0-9]{12}(?:[0-9]{3})?|[25][1-7][0-9]{14}|6(?:011|5[0-9][0-9])[0-9]{12}|3[47][0-9]{13}|3(?:0[0-5]|[68][0-9])[0-9]{11}|(?:2131|1800|35\d{3})\d{11})$, Description: creditcard
BlocklistItemId: 53e669e1-ee00-4819-8fda-6776ae2915b0, Text: ^\d{3}-?\d{2}-?\d{4}$, Description: ssn
BlocklistItemId: 2b40aa83-e411-46d8-89d0-479ac02a2d6c, Text: Alex, Description: name


### Analyze text with a blocklist

In [ ]:
import os
from azure.ai.contentsafety import ContentSafetyClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentsafety.models import AnalyzeTextOptions
from azure.core.exceptions import HttpResponseError

key = os.getenv("CONTENT_SAFETY_KEY")
endpoint = os.getenv("CONTENT_SAFETY_ENDPOINT")

# Create a Content Safety client
client = ContentSafetyClient(endpoint, AzureKeyCredential(key))

# 4111111111111111
# 123-45-6789
blocklist_name = "blocklist1"
input_text = "my ssn card number is 4111111111111111" 

try:
    # After you edit your blocklist, it usually takes effect in 5 minutes, please wait some time before analyzing
    # with blocklist after editing.
    analysis_result = client.analyze_text(
        AnalyzeTextOptions(text=input_text, blocklist_names=[blocklist_name], halt_on_blocklist_hit=False)
    )
    if analysis_result and analysis_result.blocklists_match:
        print("\nBlocklist match results: ")
        for match_result in analysis_result.blocklists_match:
            print(
                f"BlocklistName: {match_result.blocklist_name}, BlocklistItemId: {match_result.blocklist_item_id}, "
                f"BlocklistItemText: {match_result.blocklist_item_text}"
            )
except HttpResponseError as e:
    print("\nAnalyze text failed: ")
    if e.error:
        print(f"Error code: {e.error.code}")
        print(f"Error message: {e.error.message}")
        raise
    print(e)
    raise


Blocklist match results: 
BlocklistName: blocklist1, BlocklistItemId: 24f20276-7511-42cc-8cab-9d136036a0b7, BlocklistItemText: \b\d{3}-?\d{2}-?\d{4}\b
